# Shakespear RNN

## Introduction
This project aims to develop and train a simple RNN through Shakespear's books to predict the next word mostly accurately. For this project, a dataset called **Tiny Shakespear** is gonna be used, it has the particularity to be already processed.

## Importing libraries

In [25]:
import torch
import urllib.request
import matplotlib
import numpy as np
import torch.nn.functional as F

## Testing device

In [3]:
assert torch.cuda.is_available(), "GPU must be used for runtime"

## Retrieving data

In [4]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
fichier_dest = "../dataset/shakespeare.txt"

urllib.request.urlretrieve(url, fichier_dest)

with open(fichier_dest, "r", encoding="utf-8") as f:
    text = f.read()

## Constructing and vectorizing vocabulary

In [5]:
vocabulary = sorted(list(set(text)))
char_to_index = { vocabulary[i] : i for i in range(len(vocabulary)) }
index_to_char = vocabulary

def vectorize_text(text, device: torch.device) -> torch.Tensor: 
    size = len(text)
    res = torch.zeros(size).to(device)
    for i in range(size): 
        res[i] = char_to_index[text[i]]
    return res

vectorized_text = vectorize_text(text, torch.cuda.current_device())

## Creating the RNN

### Batch retrieving function

In [6]:
def get_batch(
    vectorized_text: torch.Tensor, 
    sequence_size: int, 
    batch_size: int, 
    device: torch.device
): 
    n = vectorized_text.shape[0] - 1
    idx = torch.randint(0, n - sequence_size, (batch_size,))
    input_batch = torch.stack([vectorized_text[i : i + sequence_size] for i in idx]).to(device, dtype=torch.long)
    output_batch = torch.stack([vectorized_text[i + 1 : i + sequence_size + 1] for i in idx]).to(device, dtype=torch.long)

    return input_batch, output_batch

### Subclassing the `nn.Module` class to create the RNN

In [7]:
class ShakespearRNN(torch.nn.Module): 
    _input_size: int 
    _hidden_size: int 
    _embedding_dim: int 

    _embedding: torch.nn.Embedding
    _lstm: torch.nn.LSTM
    _linear: torch.nn.Linear

    def __init__(
        self,
        input_size: int, 
        hidden_size: int,
        embedding_dim: int,
        num_layers: int = 1,
    ): 
        super(ShakespearRNN, self).__init__()

        self._input_size = input_size
        self._hidden_size = hidden_size
        self._embedding_dim = embedding_dim

        self._embedding = torch.nn.Embedding(
            num_embeddings=input_size,
            embedding_dim=embedding_dim
        )
        self._lstm = torch.nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers, 
            batch_first=True
        )
        self._linear = torch.nn.Linear(
            in_features=hidden_size,
            out_features=input_size
        )

    def init_hidden(self, batch_size: int, device: torch.device): 
        state = torch.zeros(1, batch_size, self._hidden_size).to(device)
        cell = torch.zeros(1, batch_size, self._hidden_size).to(device)

        return (state, cell)

    def forward(self, x: torch.Tensor, state: torch.Tensor = None, return_state: torch.Tensor = False):
        x_embedd = self._embedding(x)
        if state is None:
            state = self.init_hidden(x.size(0), x.device)
        out, state = self._lstm(x_embedd, state)
        out = self._linear(out)

        return out if return_state else (out, state)

### Instanciating the RNN

In [ ]:
input_size = len(vocabulary)
hidden_size = 1024
embedding_dim = 256
batch_size = 64
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sequence_size = 150

rnn = ShakespearRNN(
    input_size=input_size,
    hidden_size=hidden_size,
    embedding_dim=embedding_dim,
    num_layers=1
).to(device)

### Testing the RNN without training

In [9]:
x, y = get_batch(vectorized_text, sequence_size, batch_size, device)
(pred, state) = rnn(x)

In [10]:
sampled_indices = torch.multinomial(torch.softmax(pred[0], dim=-1), num_samples=1)
sampled_indices = sampled_indices.squeeze(-1).cpu().numpy()

x_text = "".join([index_to_char[i] for i in x[0]])
pred_text = "".join([index_to_char[i] for i in sampled_indices])

print(f"Initial text : {x_text}\n\n")
print(f"Predicted text before training : {pred_text}")

cross_entropy = torch.nn.CrossEntropyLoss()

Initial text :  beyond a prince's delicates,
His viands sparkling in a golden cup,
His body couched in a curious bed,
When care, mistrust, and treason waits on him.



Predicted text before training : -r,Y3cCRmrhqp?s.cmqLPskJpzAjWYMESCKRdlAlO?Qo!-jDZXRrYzLGU&ydb-UrtpFcfGYxvOy;Ek$o'jGJ3XLXasDpkelXFe.gSBghEK3X wiqNMdR3wR$EekF;A!O!njCm&mYcuMk.tH,g,OfOJ


## Training the network

### Computing the Cross Entropy Loss
$$
\text{CrossEntropy}(X,Y) = -\bold{E}_X(\log Y) = -\sum_{x \in \cal{X}}p_X(x) \log p_Y(x).
$$

In [11]:
def compute_loss(labels: torch.Tensor, logits: torch.Tensor) -> float: 
    batched_labels = labels.view(-1)
    batched_logits = logits.view(-1, logits.size(-1))

    return cross_entropy(batched_logits, batched_labels)

In [19]:
print(y.shape)
print(pred.shape)

example_batch_loss = compute_loss(y, pred)
print(example_batch_loss.item())

torch.Size([64, 150])
torch.Size([64, 150, 65])
1.294492244720459


### Optimizer and training function

In [21]:
params = dict(
    num_training_iterations = 10_000,
    batch_size = 32, 
    seq_length = 150,
    learning_rate = 5e-2,
    embedding_dim = 256,
    hidden_size = 1024,
)

optimizer = torch.optim.Adam(
    params=rnn.parameters(),
    lr=params["learning_rate"],
    weight_decay=1e-5
)

def training_step(x: torch.Tensor, y: torch.Tensor): 
    rnn.train()
    optimizer.zero_grad()
    y_hat, _ = rnn(x)
    loss = compute_loss(y, y_hat)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(rnn.parameters(), max_norm=5.0)
    optimizer.step()
    return loss

for iter in range(params["num_training_iterations"]): 
    x_batch, y_batch = get_batch(vectorized_text, params["seq_length"], params["batch_size"], device)
    loss = training_step(x_batch, y_batch)

    print(f"\rCurrent interation {iter}, current loss : {loss}", end="", flush=True)

Current interation 9999, current loss : 1.4264022111892763

In [35]:
def generate_text(model: ShakespearRNN, starting_string: str, generation_length: int = 1000): 
    model.eval()
    input_idx = vectorize_text(starting_string, device).unsqueeze(0).long().to(device)
    state = model.init_hidden(input_idx.size(0), device)
    text_generated = [starting_string]

    with torch.no_grad():
        for i in range(generation_length): 
            logits, state = model(input_idx, state)
            next_char_logits = logits[0, -1, :]
            probabilities = F.softmax(next_char_logits, dim=0)
            next_char_idx = torch.multinomial(probabilities, num_samples=1)
            next_char = index_to_char[next_char_idx.item()]
            text_generated.append(next_char)
            input_idx = next_char_idx.unsqueeze(0)
    return "".join(text_generated)

res = generate_text(rnn, "ROMEO:\n")

print(res)

ROMEO:
My Lord, and I waggain anon papk creat:
And lies his own so with a maid, this is
Moody prophesion of Widower, Boheme worse leave in thy drought out and a death;
And we'll abate that that had have a night.

SLY:
But mighty hop it is I never swight on to spoil fellow:
Forthy foricates of vish's trembler. Canst for you enjoy'd the night.

BENVOLIO:
Courn thee with a sonce to be men.

VINCENTIUS:
If it will not bent me heme
Ofriend, which is the doom of plug mine
Should be tender your houses, with 3 heart
The rest yet there lies away, he'll be being,
And what sue ruthly thy followers? And it, I won
promptite, with nothin couseth like your blinks:
Convice, other, by Margaret, as is again,
Punbound to confuse of all that.
Breece you wrinky with father wor instanced:
Either a cock that valiant, and flest
As in my live, who salish'd him peasure milk or endard Could I as
graved yets the lanch by send truth-seardle&s:
Peinsing oppless would be peace,
Comfort in my lior brides 'lold,
Fairl